In [ ]:
import os
import glob
import json
import numpy as np
import pandas as pd
from pathlib import Path
from PNW_cmap import PNW_cmap
import matplotlib.pyplot as plt
from vip_slap2_analysis.utils.utils import save_figure
from vip_slap2_analysis.io.session_registry import VIPSessionRegistry
from vip_slap2_analysis.glutamate.summary import GlutamateSummary
from vip_slap2_analysis.utils.utils import normalize

import seaborn as sns
sns.set_style('white')
params = {'legend.fontsize': 'x-large',
         'axes.labelsize': 'xx-large',
         'axes.titlesize':'xx-large',
         'xtick.labelsize':'xx-large',
         'ytick.labelsize':'xx-large'}
plt.rcParams.update(params)

from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
%matplotlib notebook

In [ ]:
savepath = r"C:\Users\andrew.shelton\Dropbox\allen institute\Documents\Presentations\Misc\2026-10-14_ND_NAPE_Symposium"

In [ ]:
target_mice = [804730]

registry = VIPSessionRegistry.from_basepath(
    r'\\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics'
)

process_df = registry.sessions(
    subject_ids=target_mice,
    exclude_session_types=["expression_check", "volume_imaging"],
    paradigms=["change_detection_passive"],
)

assets = [registry.resolve_assets(row) for _, row in process_df.iterrows()]

print(f"Loaded {len(assets)} session assets")

In [ ]:
asset = assets[0]
print(asset.session_id)
gs = GlutamateSummary(asset.summary_mat)

### Plot raw data from a single segment of a session for one synapse

In [ ]:
epoch_path = Path(asset.qc_dir) / 'behavior' / 'imaging_epochs.csv'
image_epochs = pd.read_csv(epoch_path)
if image_epochs.empty:
    raise ValueError(f'No imaging epochs in {epoch_path}')

In [ ]:
stim_path = Path(asset.qc_dir) / 'glutamate' / 'glutamate_extraction_qc.json'
with open(stim_path, encoding='utf-8') as f:
    im_json = json.load(f)
assert im_json.get('per_dmd'), 'No per-DMD extraction QC: run glutamate extraction first.'

In [ ]:
im_colors = [
    '#c5cae9', '#ffcdd2', '#c8e6c9', '#ffe0b2',
    '#e1bee7', '#d7ccc8',
    '#9fd3f2']

In [ ]:
# One synapse, unmodified trace, using the same epoch-aware reconstruction
# and corrected image onset times used by the processing pipeline.
from vip_slap2_analysis.common.alignment import reconstruct_dmd_session_traces

SIGNAL = im_json.get('trace_signal', 'dF')
MODE = im_json.get('trace_mode', 'ls')
IM_RATE_HZ = 200.0
STIM_DURATION_S = 0.25  # image presentation length, NOT a timing adjustment


def load_aligned_dmd(dmd):
    dmd = int(dmd)
    if dmd not in (1, 2):
        raise ValueError('DMD must be 1 or 2')
    info = im_json['per_dmd'].get(f'DMD{dmd}', {})
    if info.get('skipped') or 'stimulus_onsets_used_for_extraction' not in info:
        raise ValueError(f'DMD{dmd} has no extracted stimulus events in {stim_path}')
    bundle = reconstruct_dmd_session_traces(
        gs, dmd=dmd, im_rate_hz=IM_RATE_HZ,
        epoch_start_sec=float(image_epochs.iloc[0]['start_time']),
        signal=SIGNAL, mode=MODE,
        epoch_df=image_epochs,
        trial_epoch=getattr(gs, 'trial_epoch', None),
        strict_epoch_match=bool(im_json.get('strict_epoch_match', True)),
        min_epoch_duration_sec=float(im_json.get('min_epoch_duration_sec', 5.0)),
    )
    if bundle.traces.shape[1] != len(bundle.timebase_sec):
        raise RuntimeError('Reconstructed traces/timebase length mismatch')
    return bundle


def plot_stim_spans(ax, dmd, time_window):
    images = im_json['per_dmd'][f'DMD{dmd}']['stimulus_onsets_used_for_extraction']['image_identity']
    colors = dict(zip(images, [im_colors[i % len(im_colors)] for i in range(len(images))]))
    for image_id, onsets in images.items():
        for onset in np.unique(np.asarray(onsets, dtype=float)):
            if np.isfinite(onset) and onset < time_window[1] and onset + STIM_DURATION_S > time_window[0]:
                ax.axvspan(onset, onset + STIM_DURATION_S, color=colors[image_id], alpha=0.6, lw=0, zorder=-2)


dmd, synapse = 2, 0  # synapse indices are zero-based (raw summary indices)
time_window = (0, 1800)
bundle = load_aligned_dmd(dmd)
if not 0 <= synapse < bundle.traces.shape[0]:
    raise IndexError(f'DMD{dmd}: index {synapse} is outside 0..{bundle.traces.shape[0]-1}')
mask = (bundle.timebase_sec >= time_window[0]) & (bundle.timebase_sec <= time_window[1])
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(bundle.timebase_sec[mask], bundle.traces[synapse, mask], color='k', lw=0.8)
plot_stim_spans(ax, dmd, time_window)
ax.set(xlim=time_window, xlabel='Time (s; corrected photodiode/HARP reference)',
       ylabel='ΔF', title=f'DMD{dmd} · synapse {synapse}')
sns.despine(); fig.tight_layout()

In [ ]:
# Explicit per-row choices can combine synapses from either DMD.
SYNAPSES_TO_PLOT = [
    (1, 24),
    (1, 6),
    (1, 49),
    (2, 20),
    (2, 29),
    (2, 46),
    (2, 53),
]  # (DMD, zero-based synapse index); order determines top-to-bottom rows
TIME_WINDOW = (0, 1800)  # corrected photodiode/HARP seconds
STACK_SPACING = 1.5
TRACE_HEIGHT = 2.0
SMOOTH_WINDOW = 1  # 1 = raw/unfiltered; set >1 only when smoothing is intentional

if not SYNAPSES_TO_PLOT:
    raise ValueError('Specify at least one (DMD, synapse) pair')
cache = {dmd_: load_aligned_dmd(dmd_) for dmd_ in sorted({d for d, _ in SYNAPSES_TO_PLOT})}
fig, axes = plt.subplots(len(SYNAPSES_TO_PLOT), 1, figsize=(13, max(2.2, 1.7*len(SYNAPSES_TO_PLOT))),
                         sharex=True, squeeze=False)
axes = axes[:, 0]
for ax, (dmd_, syn_) in zip(axes, SYNAPSES_TO_PLOT):
    bundle = cache[dmd_]
    if not isinstance(syn_, (int, np.integer)) or not 0 <= syn_ < bundle.traces.shape[0]:
        raise IndexError(f'DMD{dmd_}: synapse {syn_} invalid; expected 0..{bundle.traces.shape[0]-1}')
    t = bundle.timebase_sec
    mask = (t >= TIME_WINDOW[0]) & (t <= TIME_WINDOW[1])
    y = bundle.traces[syn_, mask].copy()
    if SMOOTH_WINDOW > 1:
        y = pd.Series(y).rolling(SMOOTH_WINDOW, center=True, min_periods=1).mean().to_numpy()
    ax.plot(t[mask], y, lw=0.8, color='k')
    plot_stim_spans(ax, dmd_, TIME_WINDOW)
    ax.set_ylabel(f'DMD{dmd_}\nsyn {syn_}', rotation=0, ha='right', va='center', labelpad=22)
    ax.set_xlim(*TIME_WINDOW)
    sns.despine(ax=ax)
axes[-1].set_xlabel('Time (s; corrected photodiode/HARP reference)')
fig.tight_layout()

In [ ]:
# Same selections as the preceding plot, displayed on one axis with separate baselines.
# No smoothing by default. Robust scaling affects DISPLAY ONLY, not extracted traces.
OFFSET_SPACING = 2.0       # vertical separation between synapses
OFFSET_TRACE_HEIGHT = 1.25 # displayed size relative to robust amplitude range
OFFSET_SMOOTH_WINDOW = 10    # 1 = no smoothing
OFFSET_SHOW_RAW = False    # additionally show original trace if smoothing is enabled

fig, ax = plt.subplots(figsize=(13, 6))
y_offsets = np.arange(len(SYNAPSES_TO_PLOT)) * OFFSET_SPACING

for i, (dmd_, syn_) in enumerate(SYNAPSES_TO_PLOT):
    bundle = cache[dmd_]  # epoch-aware, processing-corrected DMD timebase
    if not 0 <= syn_ < bundle.traces.shape[0]:
        raise IndexError(f'DMD{dmd_}: invalid synapse {syn_}')
    t = bundle.timebase_sec
    mask = (t >= TIME_WINDOW[0]) & (t <= TIME_WINDOW[1])
    raw = np.asarray(bundle.traces[syn_, mask], dtype=float)
    if not np.isfinite(raw).any():
        print(f'Warning: DMD{dmd_} synapse {syn_} has no finite samples in window')
        continue
    trace = (pd.Series(raw).rolling(OFFSET_SMOOTH_WINDOW, center=True, min_periods=1)
             .mean().to_numpy() if OFFSET_SMOOTH_WINDOW > 1 else raw)
    center = np.nanmedian(trace)
    centered = trace - center
    low, high = np.nanpercentile(centered, [1, 99])
    robust_range = high - low
    if not np.isfinite(robust_range) or robust_range <= 0:
        robust_range = np.nanstd(centered)
    if not np.isfinite(robust_range) or robust_range <= 0:
        robust_range = 1.0
    y0 = y_offsets[i]
    if OFFSET_SHOW_RAW and OFFSET_SMOOTH_WINDOW > 1:
        ax.plot(t[mask], y0 + OFFSET_TRACE_HEIGHT * (raw - center) / robust_range,
                color='0.65', lw=0.7, alpha=0.5, zorder=2)
    ax.plot(t[mask], y0 + OFFSET_TRACE_HEIGHT * centered / robust_range,
            color='k', lw=1.0, zorder=3)

# Stimulus spans use extraction-corrected onsets with NO manual offset.
# If both DMDs are chosen, both sets of onsets are drawn (possibly overlapping).
for dmd_ in sorted({d for d, _ in SYNAPSES_TO_PLOT}):
    plot_stim_spans(ax, dmd_, TIME_WINDOW)

ax.set_xlim(*TIME_WINDOW)
ax.set_yticks(y_offsets)
ax.set_yticklabels([f'DMD{d} · syn {s}' for d, s in SYNAPSES_TO_PLOT])
ax.set_xlabel('Time (s; corrected photodiode/HARP reference)')
ax.set_ylabel('Selected synapses (individually scaled)')
ax.tick_params(axis='y', length=0)
for spine in ax.spines.values():
    spine.set_linewidth(1.5)
fig.tight_layout()

In [ ]:
# Same selections as the preceding plot, displayed on one axis with separate baselines.
# No smoothing by default. Robust scaling affects DISPLAY ONLY, not extracted traces.
OFFSET_SPACING = 2.0       # vertical separation between synapses
OFFSET_TRACE_HEIGHT = 1.25 # displayed size relative to robust amplitude range
OFFSET_SMOOTH_WINDOW = 10    # 1 = no smoothing
OFFSET_SHOW_RAW = False    # additionally show original trace if smoothing is enabled

fig, ax = plt.subplots(figsize=(13, 6))
y_offsets = np.arange(len(SYNAPSES_TO_PLOT)) * OFFSET_SPACING

for i, (dmd_, syn_) in enumerate(SYNAPSES_TO_PLOT):
    bundle = cache[dmd_]  # epoch-aware, processing-corrected DMD timebase
    if not 0 <= syn_ < bundle.traces.shape[0]:
        raise IndexError(f'DMD{dmd_}: invalid synapse {syn_}')
    t = bundle.timebase_sec
    mask = (t >= TIME_WINDOW[0]) & (t <= TIME_WINDOW[1])
    raw = np.asarray(bundle.traces[syn_, mask], dtype=float)
    if not np.isfinite(raw).any():
        print(f'Warning: DMD{dmd_} synapse {syn_} has no finite samples in window')
        continue
    trace = (pd.Series(raw).rolling(OFFSET_SMOOTH_WINDOW, center=True, min_periods=1)
             .mean().to_numpy() if OFFSET_SMOOTH_WINDOW > 1 else raw)
    center = np.nanmedian(trace)
    centered = trace - center
    low, high = np.nanpercentile(centered, [1, 99])
    robust_range = high - low
    if not np.isfinite(robust_range) or robust_range <= 0:
        robust_range = np.nanstd(centered)
    if not np.isfinite(robust_range) or robust_range <= 0:
        robust_range = 1.0
    y0 = y_offsets[i]
    if OFFSET_SHOW_RAW and OFFSET_SMOOTH_WINDOW > 1:
        ax.plot(t[mask], y0 + OFFSET_TRACE_HEIGHT * (raw - center) / robust_range,
                color='0.65', lw=0.7, alpha=0.5, zorder=2)
    ax.plot(t[mask], y0 + OFFSET_TRACE_HEIGHT * centered / robust_range,
            color='k', lw=1.0, zorder=3)

# Stimulus spans use extraction-corrected onsets with NO manual offset.
# If both DMDs are chosen, both sets of onsets are drawn (possibly overlapping).
for dmd_ in sorted({d for d, _ in SYNAPSES_TO_PLOT}):
    plot_stim_spans(ax, dmd_, TIME_WINDOW)

ax.set_xlim(*TIME_WINDOW)
ax.set_yticks(y_offsets)
ax.set_yticklabels([f'DMD{d} · syn {s}' for d, s in SYNAPSES_TO_PLOT])
ax.set_xlabel('Time (s; corrected photodiode/HARP reference)')
ax.set_ylabel('Selected synapses (individually scaled)')
ax.tick_params(axis='y', length=0)
for spine in ax.spines.values():
    spine.set_linewidth(1.5)
fig.tight_layout()

In [ ]:
# Same single-DMD selections as the preceding plot, displayed on one axis
# with separate baselines.
#
# No smoothing by default. Robust scaling affects DISPLAY ONLY, not extracted traces.

DMD_TO_PLOT = 2
TIME_WINDOW = (60, 76)
SYNAPSES_TO_PLOT = list(np.linspace(10,29,20).astype(int))#list(np.arange(30))

OFFSET_SPACING = 5
OFFSET_TRACE_HEIGHT = 5
OFFSET_SMOOTH_WINDOW = 2   # 1 = no smoothing
OFFSET_SHOW_RAW = False

if not SYNAPSES_TO_PLOT:
    raise ValueError("Specify at least one synapse index")

bundle = load_aligned_dmd(DMD_TO_PLOT)

# Validate requested synapses
for syn_ in SYNAPSES_TO_PLOT:
    if not isinstance(syn_, (int, np.integer)):
        raise TypeError(f"Synapse indices must be integers; got {syn_!r}")

    if not 0 <= syn_ < bundle.traces.shape[0]:
        raise IndexError(
            f"DMD{DMD_TO_PLOT}: synapse {syn_} invalid; "
            f"expected 0..{bundle.traces.shape[0] - 1}"
        )

fig, ax = plt.subplots(figsize=(12, 6))


y_offsets = np.arange(len(SYNAPSES_TO_PLOT)) * OFFSET_SPACING

t = bundle.timebase_sec
mask = (t >= TIME_WINDOW[0]) & (t <= TIME_WINDOW[1])

for i, syn_ in enumerate(SYNAPSES_TO_PLOT):

    raw = np.asarray(
        bundle.traces[syn_, mask],
        dtype=float,
    )

    if not np.isfinite(raw).any():
        print(
            f"Warning: DMD{DMD_TO_PLOT} synapse {syn_} "
            f"has no finite samples in window"
        )
        continue

    # Optional smoothing for display only
    if OFFSET_SMOOTH_WINDOW > 1:
        trace = (
            pd.Series(raw)
            .rolling(
                OFFSET_SMOOTH_WINDOW,
                center=True,
                min_periods=1,
            )
            .mean()
            .to_numpy()
        )
    else:
        trace = raw

    # Robust centering/scaling for stacked visualization only
    center = np.nanmedian(trace)
    centered = trace - center

    low, high = np.nanpercentile(centered, [1, 99])
    robust_range = high - low

    if not np.isfinite(robust_range) or robust_range <= 0:
        robust_range = np.nanstd(centered)

    if not np.isfinite(robust_range) or robust_range <= 0:
        robust_range = 1.0

    y0 = y_offsets[i]

    # Optionally show raw trace underneath smoothed trace
    if OFFSET_SHOW_RAW and OFFSET_SMOOTH_WINDOW > 1:
        raw_vis = (
            y0
            + OFFSET_TRACE_HEIGHT
            * (raw - center)
            / robust_range
        )

        ax.plot(
            pd.DataFrame(t[mask]).rolling(10,min_periods=1).mean(),
            raw_vis,
            color="k",
            lw=0.75,
            alpha=1.0,
            zorder=2,
        )

    trace_vis = (
        y0
        + OFFSET_TRACE_HEIGHT
        * centered
        / robust_range
    )

    ax.plot(
        t[mask],
        trace_vis,
        color="k",
        lw=1.25,
        zorder=3,
    )

# Stimulus spans for this DMD only
plot_stim_spans(
    ax,
    DMD_TO_PLOT,
    TIME_WINDOW,
)

# -------------------------
# Axes formatting
# -------------------------

ax.set_xlim(*TIME_WINDOW)

ax.set_yticks(y_offsets)
ax.set_yticklabels(
    SYNAPSES_TO_PLOT
)

ax.set_xlabel(
    "Time (s)"
)

ax.set_ylabel(
    f"Synapses"
)

ax.tick_params(
    axis="y",
    length=0,
)

ax.tick_params(axis='x', which='major', reset=True, top=False, labelsize=12)
ax.tick_params(axis='y', which='major', reset=True, right=False, labelsize=12)

for spine in ax.spines.values():
    spine.set_linewidth(1.5)
ax.set_axis_off()
fig.tight_layout()
filen = f'raw_synapses'
save_figure(fig,os.path.join(savepath,filen),formats=['.pdf'],dpi=300)

### Plot mean image responses

In [ ]:
gs_mean_path = glob.glob(os.path.join(asset.derived_dir,'glutamate','glutamate_mean_df.npz'))[0]
mean_data = np.load(gs_mean_path,allow_pickle=True)['data'][0][f'DMD{dmd}']['image_identity']

In [ ]:
im_names = list(mean_data.keys())
im_names

In [ ]:
synapse = 13

fig,ax=plt.subplots(figsize=(4,4))

sns.despine()

ax.tick_params(axis='x', which='major', reset=True, top=False, labelsize=12)
ax.tick_params(axis='y', which='major', reset=True, right=False, labelsize=12)

trial_shape = mean_data[im_names[0]]['mean'][synapse].shape[0]
time = np.linspace(-0.25,0.5,trial_shape)

for i,im in enumerate(im_names):
    syn_data = mean_data[im]['mean'][synapse]
    im_rolling = pd.DataFrame(syn_data).rolling(1,min_periods=1).mean()
    ax.plot(time,im_rolling,color=im_colors[i],lw=2)
    
mean = np.mean([mean_data[im]['mean'][synapse] for im in im_names],axis=0)
rolling_mean  = pd.DataFrame(mean).rolling(1,min_periods=1).mean()

ax.plot(time,mean,color='k',lw=3)
ax.axvspan(-0.01,0.25,color='lightgray',alpha=0.4,zorder=0)

ax.set_xlabel('Time (s) from stimulus onset')
ax.set_ylabel('\u0394F',rotation=90,labelpad=0)

for spine in ['left','bottom']:
    ax.spines[spine].set_linewidth(2)

fig.tight_layout()

filen = f'synapse{synapse}_image responses'
save_figure(fig,os.path.join(savepath,filen),formats=['.pdf'],dpi=300)

### Plot the response of a given synapse to a single trial

In [ ]:
gs_trial_path = glob.glob(os.path.join(asset.derived_dir,'glutamate','glutamate_single_trial_df.npz'))[0]

In [ ]:
trial_data = np.load(gs_trial_path,allow_pickle=True)['data'][0][f'DMD{dmd}']['image_identity']

In [ ]:
im_names = list(trial_data.keys())
im_names

In [ ]:
i = 4
synapse_trials = trial_data[im_names[i]][:,synapse,:]

trial_shape = synapse_trials.shape[1]

# tr= 250
fs = 200
for ii,tr in enumerate(np.arange(219,222)):
    fig,ax=plt.subplots(figsize=(4,4))

    sns.despine(left=True)
    
    ax.tick_params(axis='x', which='major', reset=True, top=False, labelsize=12)
    ax.tick_params(axis='y', which='major', reset=True, right=False, labelsize=12)
    
    time = np.linspace(-0.25,0.5,trial_shape)

    rolling = pd.DataFrame(synapse_trials[tr]).rolling(5,min_periods=1).mean()

    ax.plot(time,rolling,color='k',lw=2)

    ax.set_yticks([])

    ax.set_xlabel('Time (s) from stim. onset')

    ax.spines['bottom'].set_linewidth(2)

    ax.axvspan(0.0,0.25,color=im_colors[i],alpha=0.5,zorder=0)
    ax.set_ylabel(f'Trial {ii+1}',rotation=0,labelpad=25)
    fig.tight_layout()
    filen = f'example_Trial{ii}'
    save_figure(fig,os.path.join(savepath,filen),formats = ['.pdf'],dpi=300)

In [ ]:
i = 4
synapse_trials = trial_data[im_names[i]][:,synapse,:]
trial_shape = synapse_trials.shape[1]

# tr= 250
fs = 200
fig,ax=plt.subplots(figsize=(4,4))

sns.despine(left=True)

ax.tick_params(axis='x', which='major', reset=True, top=False, labelsize=12)
ax.tick_params(axis='y', which='major', reset=True, right=False, labelsize=12)

time = np.linspace(-0.25,0.5,trial_shape)

mean = np.nanmean(synapse_trials,axis=0)

rolling = pd.DataFrame(mean).rolling(5,min_periods=1).mean()

ax.plot(time,rolling,color='k',lw=2)

ax.set_yticks([])

ax.set_xlabel('Time (s) from stim. onset')

ax.spines['bottom'].set_linewidth(2)

ax.axvspan(0.0,0.25,color=im_colors[i],alpha=0.5,zorder=0)
ax.set_ylabel(f'Trial {ii+1}',rotation=0,labelpad=25)
fig.tight_layout()
filen = f'example_mean'
save_figure(fig,os.path.join(savepath,filen),formats = ['.pdf'],dpi=300)

In [ ]:
i = 4
synapse_trials = trial_data[im_names[i]][:,synapse,:]
trial_shape = synapse_trials.shape[1]
fig,ax=plt.subplots(figsize=(4,4))
time = np.linspace(-0.25,0.5,trial_shape)
# tr= 250
fs = 200
for ii,tr in enumerate(synapse_trials):

    sns.despine(left=True)
    
    ax.tick_params(axis='x', which='major', reset=True, top=False, labelsize=12)
    ax.tick_params(axis='y', which='major', reset=True, right=False, labelsize=12)

    rolling = pd.DataFrame(synapse_trials[ii]).rolling(5,min_periods=1).mean()

    ax.plot(time,rolling,color=im_colors[i],lw=1,alpha=0.3)

mean = np.nanmean(synapse_trials,axis=0)
rolling_mean = pd.DataFrame(mean).rolling(5,min_periods=1).mean()

ax.plot(time,rolling_mean,color='k',lw=2)

ax.set_ylabel('\u0394F',rotation=0,labelpad=15)
ax.set_xlabel('Time (s) from stim. onset')

ax.axvline(0.0,color='k',dashes=[5,5])

# ax.set_ylim(0,500)

for spine in ['left','bottom']:
    ax.spines[spine].set_linewidth(2)
    
fig.tight_layout()
filen = f'example_im{i}'
# save_figure(fig,os.path.join(savepath,filen),formats = ['.pdf'],dpi=300)

### Plot dF/F for a given synapses' mean responses

In [ ]:
gs_dff_path = glob.glob(os.path.join(asset.derived_dir,'glutamate','glutamate_mean_dff_ls.npz'))[0]

In [ ]:
dff_data = np.load(gs_dff_path,allow_pickle=True)['data'][0][f'DMD{dmd}']['image_identity']

In [ ]:
im_names = list(dff_data.keys())
im_names

In [ ]:
synapse = 40

fig,ax=plt.subplots(figsize=(4,4))

sns.despine()

ax.tick_params(axis='x', which='major', reset=True, top=False, labelsize=12)
ax.tick_params(axis='y', which='major', reset=True, right=False, labelsize=12)

trial_shape = dff_data[im_names[0]]['mean'][synapse].shape[0]
time = np.linspace(-0.25,0.5,trial_shape)

for i,im in enumerate(im_names):
    syn_data = dff_data[im]['mean'][synapse]
    im_rolling = pd.DataFrame(syn_data).rolling(1,min_periods=1).mean()
    ax.plot(time,im_rolling,color=im_colors[i],lw=2)
    
mean = np.mean([dff_data[im]['mean'][synapse] for im in im_names],axis=0)
rolling_mean  = pd.DataFrame(mean).rolling(1,min_periods=1).mean()

ax.plot(time,mean,color='k',lw=3)
ax.axvspan(-0.01,0.25,color='lightgray',alpha=0.4,zorder=0)

ax.set_xlabel('Time (s) from stimulus onset')
ax.set_ylabel('\u0394F/F$_{0}$',rotation=90,labelpad=0)

for spine in ['left','bottom']:
    ax.spines[spine].set_linewidth(2)

fig.tight_layout()

In [ ]:
mean